In [4]:
import os
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d

In [9]:
root_dir = 'data_set'

columns_merged = [
    'Left_Hallux_raw', 'Right_Hallux_raw',
    'Left_Toes_raw', 'Right_Toes_raw',
    'Left_Met1_raw', 'Left_Met3_raw', 'Left_Met5_raw',
    'Right_Met1_raw', 'Right_Met3_raw', 'Right_Met5_raw', 
    'Left_Arch_raw', 'Right_Arch_raw',
    'Left_Heel_R_raw', 'Left_Heel_L_raw', 
    'Right_Heel_L_raw', 'Right_Heel_R_raw',

    "acceleration_Pelvis_x_local","acceleration_Pelvis_y_local","acceleration_Pelvis_z_local",
    "acceleration_RightForeArm_x_local","acceleration_RightForeArm_y_local","acceleration_RightForeArm_z_local",
    "acceleration_RightUpperLeg_x_local","acceleration_RightUpperLeg_y_local","acceleration_RightUpperLeg_z_local",
    "acceleration_RightLowerLeg_x_local","acceleration_RightLowerLeg_y_local","acceleration_RightLowerLeg_z_local",
    "acceleration_RightFoot_x_local","acceleration_RightFoot_y_local","acceleration_RightFoot_z_local",
    "acceleration_RightToe_x_local","acceleration_RightToe_y_local","acceleration_RightToe_z_local",
    "acceleration_LeftUpperLeg_x_local","acceleration_LeftUpperLeg_y_local","acceleration_LeftUpperLeg_z_local",
    "acceleration_LeftLowerLeg_x_local","acceleration_LeftLowerLeg_y_local","acceleration_LeftLowerLeg_z_local",
    "acceleration_LeftFoot_x_local","acceleration_LeftFoot_y_local","acceleration_LeftFoot_z_local",
    "acceleration_LeftToe_x_local","acceleration_LeftToe_y_local","acceleration_LeftToe_z_local",

    "angularVelocity_Pelvis_x_local","angularVelocity_Pelvis_y_local","angularVelocity_Pelvis_z_local",
    "angularVelocity_RightForeArm_x_local","angularVelocity_RightForeArm_y_local","angularVelocity_RightForeArm_z_local",
    "angularVelocity_RightUpperLeg_x_local","angularVelocity_RightUpperLeg_y_local","angularVelocity_RightUpperLeg_z_local",
    "angularVelocity_RightLowerLeg_x_local","angularVelocity_RightLowerLeg_y_local","angularVelocity_RightLowerLeg_z_local",
    "angularVelocity_RightFoot_x_local","angularVelocity_RightFoot_y_local","angularVelocity_RightFoot_z_local",
    "angularVelocity_RightToe_x_local","angularVelocity_RightToe_y_local","angularVelocity_RightToe_z_local",
    "angularVelocity_LeftUpperLeg_x_local","angularVelocity_LeftUpperLeg_y_local","angularVelocity_LeftUpperLeg_z_local",
    "angularVelocity_LeftLowerLeg_x_local","angularVelocity_LeftLowerLeg_y_local","angularVelocity_LeftLowerLeg_z_local",
    "angularVelocity_LeftFoot_x_local","angularVelocity_LeftFoot_y_local","angularVelocity_LeftFoot_z_local",
    "angularVelocity_LeftToe_x_local","angularVelocity_LeftToe_y_local","angularVelocity_LeftToe_z_local",

    'participant_id',  'walk_mode'
]

columns_annotations = columns_merged + ['stepcount']

# Sliding window parameters
window_size = 120  # 2 seconds @ 60Hz
step_size = 60     # 1 second

In [3]:
def sliding_window_segment(df):
    windows = []
    window_number = 1
    for start in range(0, len(df) - window_size + 1, step_size):
        window = df.iloc[start:start + window_size].copy()
        if window['walk_mode'].nunique() == 1:
            window['window_number'] = window_number
            windows.append(window)
            window_number += 1
    return pd.concat(windows, ignore_index=True) if windows else pd.DataFrame()

In [ ]:
print("Trimming data set...")
for course in ['courseA', 'courseB', 'courseC']:
    course_path = os.path.join(root_dir, course)
    for participant in os.listdir(course_path):
        participant_path = os.path.join(course_path, participant)
        print("Working on", participant_path)
        merged_path = os.path.join(participant_path, 'merged.csv')
        annot_path = os.path.join(participant_path, 'merged_gait_count_annotations.csv')

        if os.path.isfile(merged_path):
            df_merged = pd.read_csv(merged_path)
            df_merged = df_merged[columns_merged]
            segmented = sliding_window_segment(df_merged)
            segmented.to_csv(os.path.join(participant_path, 'sliding_window.csv'), index=False)

        if os.path.isfile(annot_path):
            df_annot = pd.read_csv(annot_path)
            df_annot = df_annot[columns_annotations]
            df_annot.to_csv(os.path.join(participant_path, 'gait_segmentation.csv'), index=False)
print("Data set trimmed successfully")

In [ ]:
print("Merging data files...")
gait_segmentation_dfs = []
sliding_window_dfs = []

for course in ['courseA', 'courseB', 'courseC']:
    course_path = os.path.join(root_dir, course)
    for participant in os.listdir(course_path):
        participant_path = os.path.join(course_path, participant)
        print("Merging", participant_path)
        gait_seg_path = os.path.join(participant_path, 'gait_segmentation.csv')
        sliding_window_path = os.path.join(participant_path, 'sliding_window.csv')

        if os.path.isfile(gait_seg_path):
            df_gait = pd.read_csv(gait_seg_path)
            gait_segmentation_dfs.append(df_gait)

        if os.path.isfile(sliding_window_path):
            df_slide = pd.read_csv(sliding_window_path)
            sliding_window_dfs.append(df_slide)

if gait_segmentation_dfs:
    merged_gait = pd.concat(gait_segmentation_dfs, ignore_index=True)
    merged_gait.to_csv(os.path.join(root_dir, 'merged_gait_segmentation.csv'), index=False)

if sliding_window_dfs:
    merged_slide = pd.concat(sliding_window_dfs, ignore_index=True)
    merged_slide.to_csv(os.path.join(root_dir, 'merged_sliding_window.csv'), index=False)

print("Data files merged successfully")

In [ ]:
df_sw = pd.read_csv(os.path.join(root_dir, "merged_sliding_window.csv"))

new_window_ids = []
group_id = 0
last_id = None

for current_id in df_sw['window_number']:
    if current_id != last_id:
        group_id += 1
    new_window_ids.append(group_id)
    last_id = current_id

df_sw['window_number'] = new_window_ids
df_sw.to_csv(os.path.join(root_dir, "merged_sliding_window.csv"), index=False)
print("Updated 'window_number' in merged_sliding_window.csv")

df_gait = pd.read_csv(os.path.join(root_dir, "merged_gait_segmentation.csv"))

new_stepcounts = []
group_id = 0
last_step = None

for current_step in df_gait['stepcount']:
    if current_step != last_step:
        group_id += 1
    new_stepcounts.append(group_id)
    last_step = current_step

df_gait['stepcount'] = new_stepcounts
df_gait.to_csv(os.path.join(root_dir, "merged_gait_segmentation.csv"), index=False)
print("Updated 'stepcount' in merged_gait_segmentation.csv")

In [ ]:
# Interpolate gait cycles into 100 rows each
df = pd.read_csv(os.path.join(root_dir, "merged_gait_segmentation.csv"))

# Metadata columns that should remain constant within each gait cycle
metadata_cols = ['participant_id', 'walk_mode', 'stepcount']

# All other columns are numerical data to be interpolated
data_cols = [col for col in df.columns if col not in metadata_cols]

interpolated_cycles = []

for (participant_id, walk_mode, stepcount), group in df.groupby(metadata_cols):
    group = group.reset_index(drop=True)
    n_rows = len(group)
    
    original_index = np.linspace(0, 1, n_rows)
    target_index = np.linspace(0, 1, 100)
    
    interpolated_data = {}
    
    # Interpolate each data column
    for col in data_cols:
        values = group[col].values
        f = interp1d(original_index, values, kind='linear', fill_value='extrapolate')
        interpolated_data[col] = f(target_index)

    interpolated_data['participant_id'] = [participant_id] * 100
    interpolated_data['walk_mode'] = [walk_mode] * 100
    interpolated_data['stepcount'] = [stepcount] * 100
    
    interpolated_cycles.append(pd.DataFrame(interpolated_data))

# Concatenate all interpolated gait cycles
interpolated_df = pd.concat(interpolated_cycles, ignore_index=True)

interpolated_df.to_csv(os.path.join(root_dir, "merged_gait_segmentation_interpolated.csv"), index=False)

print("Interpolation complete. Saved to 'merged_gait_segmentation_interpolated.csv'")

In [ ]:
df = pd.read_csv(os.path.join(root_dir, "merged_gait_segmentation_interpolated.csv"))
print(df.info())

In [ ]:
df = pd.read_csv(os.path.join(root_dir, "merged_sliding_window.csv"))
print(df.info())